Before you turn this problem in, make sure everything runs as expected. First, **restart the kernel** (in the menubar, select Kernel$\rightarrow$Restart) and then **run all cells** (in the menubar, select Cell$\rightarrow$Run All).

Make sure you fill in any place that says `YOUR CODE HERE` or "YOUR ANSWER HERE", as well as your name and collaborators below:

In [ ]:
NAME = ""
COLLABORATORS = ""

---

# Problem description

Satellites orbiting the Earth take photos that private companies sell to customers. A financial analyst might extract information from these photos that helps predict the future price of a stock, for example:

- Approximate number of customers visiting each store: count the cars in the parking lot
- Approximate activity in a factory: count supplier trucks arriving and delivery trucks leaving
- Approximate demand for a commodity: count cargo ships traveling between ports

In this assignment we will recognize **ships in satellite photos**: a first step toward counting.
We ignore domain-specific knowledge and just use a labeled training set (a photo plus a binary indicator for whether a ship is present), assuming the labels are perfect.

This is the same problem as the earlier *Ships in satellite images* assignment, where you used Logistic Regression.
This time you will use a **neural network**, and you will run a set of experiments to build intuition for how
the **number of layers** and the **width** of the layers affect the model.

## Goal

Build and compare **Dense** (fully-connected) neural-network classifiers for satellite photos.

- The features are images: 80 x 80 pixels (we will convert to gray scale and flatten, exactly as in the Logistic Regression assignment)
- The labels are 1 (ship is present) or 0 (ship is not present)

## Learning objectives
- See that a neural network with no hidden layer *is* Logistic Regression
- Count the parameters of a Dense network
- Build intuition for the effect of depth (number of layers, $L$) and width (units per layer) on accuracy and on the gap between in-sample and out-of-sample performance
- Practice disciplined model selection: choose on a validation set, touch the test set once

## This is a checkpoint assignment
It verifies that you can build, train and compare neural networks. It is not meant to be a puzzle.
Most of the work is running experiments and **thinking about what you see**.

## Grading
The required parts are worth **200 points**. You can earn up to **100 additional points** for going above and beyond: the optional extensions at the end of the notebook and any further experiments of your own. In the required parts, written answers are graded on whether they are consistent with **your own results** and give specific support (a number from your table or a named configuration), not on length. Optional work is graded on whether it ran, on what you conclude from it, and on how carefully you handled the noise.

## How to run this notebook

**Toolkit:** use any neural-network toolkit you like (Keras with any backend, TensorFlow, PyTorch, ...).
Everything you are asked to *report* is toolkit-independent (numbers, tables, short written answers).

**Hardware:** you do **not** need a GPU. The whole notebook should run in a few minutes on an ordinary laptop CPU.

**Running locally:** put `helper.py` and the data file `shipsnet.json` next to this notebook (the data may also be in a `Data/` folder).

**Running on Google Colab:** (optional, but fine)
- Upload `helper.py` to **the same directory as your notebook**
- **Recommended: put `shipsnet.json` (344 MB) on your own Google Drive, once.** Upload it to `My Drive/Colab Notebooks/Data/` (from drive.google.com or a Colab session) and mount the Drive in a cell (`from google.colab import drive; drive.mount('/content/drive')`; Google will ask for your permission). The helper looks in `Colab Notebooks/Data` and `Data` on your Drive automatically; if you used another folder, set `DATA_DIR` (next cell group) to the folder that contains `shipsnet.json`
- Uploading `shipsnet.json` to the Colab session through the *Files* panel also works in principle, but it is a 344 MB upload (slow, and it can fail on a poor connection) and the file disappears whenever the runtime is recycled, so you would have to upload it again
  - If the data is on your Drive in another folder, set `DATA_DIR` (next cell group) to the folder that contains `shipsnet.json`
  - If the notebook itself is on your Drive, mount the Drive and `%cd` into the folder that contains the notebook and `helper.py`

# Imports

In [ ]:
## Standard imports
import os
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from IPython.display import display

%matplotlib inline

In [ ]:
## Are we running on Colab?
try:
  from google.colab import drive
  IN_COLAB=True
except:
  IN_COLAB=False

if IN_COLAB:
  print("We're running Colab")

In [ ]:
## Where is the data?
# Leave as None if shipsnet.json is next to this notebook (or in ./Data).
# Otherwise set it to the folder that contains shipsnet.json, e.g. (Colab + Google Drive):
#   drive.mount('/content/drive')
#   DATA_DIR = "/content/drive/MyDrive/my_folder"
DATA_DIR = None

In [ ]:
## Load the helper module
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Optional convenience: pick up edits to helper.py without restarting the kernel.
# (Not available on every Python/IPython combination, e.g. Colab's, so failure is fine.)
try:
    get_ipython().run_line_magic("reload_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "1")
    AUTORELOAD = True
except Exception:
    AUTORELOAD = False

import helper
if AUTORELOAD:
    get_ipython().run_line_magic("aimport", "helper")

helper = helper.Helper(data_dir=DATA_DIR)

In [ ]:
## Import your neural-network toolkit here (plus anything else you need)
# Tip (TensorFlow/Keras): the harmless C++ log messages can be silenced by setting
#   os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   before importing keras/tensorflow
# YOUR CODE HERE
raise NotImplementedError()

# API for students

`helper.py` defines a class `Helper`; `helper = helper.Helper()` creates one.

- `getData`: get a collection of labeled images

  >`data, labels = helper.getData()`
- `showData`: visualize labeled images

  >`helper.showData(data, labels)`
- `showWeights`: visualize weight vectors as images (used in an *optional* part at the end)

  >`helper.showWeights(W)`

# Get the data

The first step in our Recipe is Get the Data. (Loading the 344 MB file takes 20 to 30 seconds.)

In [ ]:
# Get the data
data, labels = helper.getData()
n_samples, width, height, channel = data.shape

print("Data shape: ", data.shape)
print("Labels shape: ", labels.shape)
print("Label values: ", np.unique(labels))

Your expected outputs should be the following

`Data shape: (4000, 80, 80, 3)`
`Labels shape: (4000,)`
`Label values: [0 1]`

We shuffle the examples before doing anything else (the file may be ordered by label).

In [ ]:
# Shuffle the data first
data, labels = sklearn.utils.shuffle(data, labels, random_state=42)

## Have a look at the data
We will not go through all steps in the Recipe, nor in depth. But here's a peek.

In [ ]:
# Visualize some samples
helper.showData(data[:25], labels[:25])

## Eliminate the color dimension and flatten
As in the Logistic Regression assignment, we convert to gray scale.
Then we **flatten** each 80 x 80 image into a vector of 6400 numbers, since a Dense layer takes a vector as input.

In [ ]:
print("Original shape of data: ", data.shape)

w = (.299, .587, .114)
data_bw = np.sum(data * w, axis=3)

# Flatten each image: (N, 80, 80) --> (N, 6400)
X = data_bw.reshape(data_bw.shape[0], -1)
y = labels

print("New shape of data: ", data_bw.shape)
print("Flattened X shape: ", X.shape)

## Split the data: train / validation / test

In the Logistic Regression assignment you split 90% / 10% (train / test).
Here you will run **many** experiments and *choose among them*. If you chose using the test set, the test accuracy of the model you
chose would be optimistic (you would have "trained on the test set" by selection).
So we carve out a **validation** set to choose with, and we touch the **test** set only **once**, at the very end.

**Question:** create `X_train, X_val, X_test, y_train, y_val, y_test` as follows:
- First split off 10% of the examples as the **test** set, with `random_state=42`
  - (This is exactly the same test set as in the Logistic Regression assignment)
- Then split the remaining 90% into train and validation so that the **validation** set has 400 examples (that is, 1/9 of what remains), with `random_state=42`

You should end up with 3200 training, 400 validation and 400 test examples.

In [ ]:
from sklearn.model_selection import train_test_split

# Create X_train, X_val, X_test, y_train, y_val, y_test
# YOUR CODE HERE
raise NotImplementedError()

print("X_train shape: ", X_train.shape)
print("X_val shape:   ", X_val.shape)
print("X_test shape:  ", X_test.shape)

# Self-check (given): sizes, and number of ships in each set
print("ships in train/val/test:", int(y_train.sum()), int(y_val.sum()), int(y_test.sum()))
assert (X_train.shape[0], X_val.shape[0], X_test.shape[0]) == (3200, 400, 400)
assert (int(y_train.sum()), int(y_val.sum()), int(y_test.sum())) == (812, 101, 87), "Right sizes but different examples: follow the order of the two splits described above"


Your expected outputs should be the following

`X_train shape:  (3200, 6400)`
`X_val shape:    (400, 6400)`
`X_test shape:   (400, 6400)`

## Standardize

**Question:** standardize the pixels (mean 0, standard deviation 1 per pixel position), as you did before.
- Fit a `StandardScaler` named `scaler` on the **training** examples only
- Create `X_train_s`, `X_val_s`, `X_test_s`: the scaled versions of the three sets

(From here on, every model in this notebook uses these scaled arrays.)

In [ ]:
from sklearn.preprocessing import StandardScaler

# YOUR CODE HERE
raise NotImplementedError()

---
# Part 1: The Logistic Regression baseline

Re-create the baseline from the earlier assignment so that we have something to compare the neural networks against.

**Question:**
- Fit an `sklearn` `LogisticRegression(solver="liblinear")` on the scaled training examples. Name it `lr_clf`.
- Set `lr_train_acc` and `lr_val_acc` to its accuracy on the training and validation examples
- Set `lr_num_params` to its **total** number of learnable parameters (**including** the intercept)

*(Do not evaluate on the test set yet.)*

Sanity check: the validation accuracy should land between about 0.82 and 0.90.

In [ ]:
from sklearn.linear_model import LogisticRegression

# YOUR CODE HERE
raise NotImplementedError()

print("Logistic Regression: train acc={a:.3f}, validation acc={v:.3f}, #parameters={p:d}".format(
    a=lr_train_acc, v=lr_val_acc, p=lr_num_params))

---
# Part 2: A Dense network with no hidden layer

Define **$L$ = the number of *hidden* layers**. The output layer is not counted.

The simplest Dense classifier has $L = 0$: the 6400 inputs connect directly to a single output unit with a **sigmoid** activation,
trained with **binary cross-entropy** loss.

## The training recipe (the same for *every* network in this notebook)

So that differences between experiments come from the **architecture** and not from the training procedure, always use:

> **Read this table before you write any code, and check your code against it afterwards.** Every row is part of the experiment. The **random seed** and **batch shuffling** rows are the two that are easiest to get wrong, and they matter for the noise questions later.

| Setting | Value |
|---|---|
| Loss | binary cross-entropy |
| Optimizer | Adam, learning rate 0.001 |
| Batch size | 64 |
| Epochs | 30 (always train all 30; no early stopping) |
| Hidden-layer activation | ReLU |
| Initialization, other optimizer settings | your toolkit's defaults |
| Output layer | 1 unit, sigmoid |
| **Random seed** | `SEED` (below). Set it the way your toolkit does, e.g. `keras.utils.set_random_seed(SEED)` or `torch.manual_seed(SEED)`. Results are then reproducible on your machine but can differ slightly on other machines |
| Inputs | the standardized gray-scale vectors `X_train_s`, ... |
| **Batches** | **shuffle the training examples every epoch** (reshuffle, not one fixed order); keep the last, smaller batch |
| Reported accuracy | evaluate the **final trained model** on the whole set (not a running average printed during training) |

**Before you rely on `SEED`:** work out what in *your* toolkit's training code is random, and check that your seed setting really controls all of it. Optional C asks you to write this down.

An example is predicted to have a ship when the output probability is greater than 0.5; *accuracy* is the fraction of examples predicted correctly.

In [ ]:
SEED = 0
EPOCHS = 30
BATCH_SIZE = 64
LEARNING_RATE = 1e-3

**Question:** build and train the $L=0$ network on the training examples, using the recipe.
- Name the model `model_L0`
- Set `L0_num_params` to the number of parameters of `model_L0` (your toolkit can report this)
- Set `L0_train_acc` and `L0_val_acc` to its final accuracy on the training and validation examples

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

print("L=0 network: train acc={a:.3f}, validation acc={v:.3f}, #parameters={p:d}".format(
    a=L0_train_acc, v=L0_val_acc, p=L0_num_params))

**Question (written):** Compare the $L=0$ network with the Logistic Regression of Part 1.
- Why is a Dense network with no hidden layer, one sigmoid output and binary cross-entropy loss "the same model" as Logistic Regression? Compare the number of parameters.
- Are the accuracies identical? Compare the **training** accuracies as well as the validation accuracies. If they differ, give a reason the two could differ even though the model family is the same (hint: look at the default settings of `LogisticRegression`, and at how each model was trained). A reasoned hypothesis is enough; you do not have to test it.

YOUR ANSWER HERE

---
# Part 3: Count parameters, then run the experiments

Now add hidden layers. Use hidden layers that all have the **same width** (number of units): `hidden_widths = [w] * L`.

## Count the parameters by hand first

A Dense layer with `n_in` inputs and `n_out` units has `n_in * n_out` weights plus `n_out` biases.

**Question:** write `count_dense_params(hidden_widths, n_inputs=6400)`, which returns the total number of parameters of a Dense network
with the given list of hidden-layer widths followed by a **single** sigmoid output unit. For example `hidden_widths=[]` is the $L=0$ network.
Compute it from the formula above: **do not build a model to get the answer**.

In [ ]:
def count_dense_params(hidden_widths, n_inputs=6400):
    """Number of parameters of a Dense network: n_inputs -> hidden_widths[0] -> ... -> hidden_widths[-1] -> 1"""
    # YOUR CODE HERE
    raise NotImplementedError()

assert count_dense_params([]) == L0_num_params, "L=0 should match your toolkit's count"
print("L=0:", count_dense_params([]))

## One function to run an experiment

**Question:** write `run_experiment(hidden_widths, seed=SEED)` that
1. builds a Dense network with the given hidden widths (ReLU) and a single sigmoid output unit,
2. trains it with the recipe on `X_train_s, y_train`,
3. returns a dictionary with keys `"model"`, `"num_params"` (reported by your toolkit), `"train_acc"` and `"val_acc"` (final accuracy on the training and validation examples).

Set the random seed at the start so that the experiment is reproducible.

In [ ]:
def run_experiment(hidden_widths, seed=SEED):
    # YOUR CODE HERE
    raise NotImplementedError()

# Sanity check: with no hidden layers this should reproduce Part 2
info = run_experiment([])
print("L=0 via run_experiment: train acc={a:.3f}, val acc={v:.3f}, #parameters={p:d}".format(
    a=info["train_acc"], v=info["val_acc"], p=info["num_params"]))

## The experiment grid

Run **every combination** of
- $L \in \{1, 2, 3\}$ hidden layers
- width $\in \{4, 16, 64, 256\}$ units in each hidden layer

That is 12 networks. For each, record the number of parameters, the training accuracy and the validation accuracy.

**Question:** create a `pandas` DataFrame named `results` with one row per experiment and the columns
`L`, `width`, `num_params`, `train_acc`, `val_acc`; also add a first row for the $L=0$ network (use `width = 0`).
Before you run each experiment, compute `count_dense_params` and **assert** that it equals the number of parameters your toolkit reports.

**Notes**
- Expect a few minutes on a laptop CPU in total (the widest networks are the slowest). It's a good idea to print progress as you go.
- If your machine is far too slow, you may drop width 256 *and say so explicitly* in your answers.
- Don't tune anything: the recipe is fixed so that only $L$ and width change.
- **Noise:** training a network involves randomness (initial weights, batch order), and the validation set has only 400 examples.
  Differences in accuracy of **about 1.5 percentage points or less are within noise**. Don't read meaning into them.
  The very narrowest networks can be especially erratic from one run to another. Expect an outlier or two in your grid (a configuration that dips for no obvious reason). If you see one, say so in your answers instead of building a story around it.

In [ ]:
rows = []

# L = 0 baseline row (from Part 2)
rows.append(dict(L=0, width=0, num_params=L0_num_params, train_acc=L0_train_acc, val_acc=L0_val_acc))

# YOUR CODE HERE
raise NotImplementedError()

results = pd.DataFrame(rows)
results

## Look at the results

The next cells (already written) plot your `results`.

In [ ]:
results["gap"] = results["train_acc"] - results["val_acc"]
display(results.round(3))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for L, grp in results[results.L > 0].groupby("L"):
    axes[0].plot(grp.width, grp.val_acc, marker="o", label="L={L}".format(L=L))
    axes[1].plot(grp.width, grp.gap, marker="o", label="L={L}".format(L=L))
    axes[2].plot(grp.num_params, grp.val_acc, marker="o", label="L={L}".format(L=L))
for ax in axes[:2]:
    ax.set_xscale("log"); ax.set_xlabel("hidden-layer width")
axes[2].set_xscale("log"); axes[2].set_xlabel("number of parameters")
axes[0].set_ylabel("validation accuracy"); axes[1].set_ylabel("train acc - validation acc"); axes[2].set_ylabel("validation accuracy")
axes[0].axhline(lr_val_acc, color="k", ls="--", lw=1, label="Logistic Regression")
axes[2].axhline(lr_val_acc, color="k", ls="--", lw=1, label="Logistic Regression")
axes[0].legend(); axes[2].legend()
fig.tight_layout();

**Questions (written).** Use your table and plots, and remember the noise warning. A short paragraph per question is plenty; refer to specific rows of your table.

1. **Width.** What happens to validation accuracy as width increases (for a fixed $L$)? Is there a point of diminishing returns?
2. **Depth.** What happens as $L$ increases (for a fixed width)? Does depth matter as much as width?
3. **Parameters.** Which grows the number of parameters faster, adding width or adding a layer? Explain using your parameter formula (which term dominates?).
4. **Where do the parameters go?** (a) Pick two configurations with the **same width** but different $L$: how different are their parameter counts, and how different are their accuracies? (b) Pick two configurations whose parameter counts differ by a **large factor** (say 10x or more) and compare their accuracies. What do you conclude about spending parameters on depth versus on width?
5. **In-sample versus out-of-sample.** How does the training accuracy compare with the validation accuracy? How does the gap change with $L$ and width? Compare the number of parameters with the number of training examples (3200). (What you see may surprise you; say what you make of it.)
6. **Versus Logistic Regression.** Does any network clearly beat Logistic Regression? By how much, and is that larger than the noise?

YOUR ANSWER HERE

---
# Part 4: Choose a model, then use the test set once

**Question:**
- Choose the configuration (`best_L`, `best_width`) among the 12 networks with at least one hidden layer ($L \geq 1$; the $L=0$ row is **not** a candidate).
  Count the correct validation predictions of each: `n_correct = round(val_acc * 400)`. Let `top` be the largest `n_correct` among them. Every configuration with `n_correct >= top - 6` is a candidate (6 examples is 1.5 points of 400; counting examples avoids floating-point ties); of these, choose the one with the **fewest parameters**.
- Train that configuration with `run_experiment` and evaluate it on the **test** set: set `test_acc`.
- Also set `lr_test_acc`, the test accuracy of your Logistic Regression `lr_clf`.

This is the **only** time in this notebook you may evaluate on the test set.

Because of noise, your chosen configuration may differ from a classmate's; what matters is that it follows the rule when applied to **your** table.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

print("Chosen network: L={L}, width={w}".format(L=best_L, w=best_width))
print("Test accuracy: chosen network={n:.3f}, Logistic Regression={l:.3f}".format(n=test_acc, l=lr_test_acc))

**Questions (written):**
1. Is the test accuracy of your chosen network similar to its validation accuracy? Would you expect it to be a little lower, and why?
2. The hidden layers raise accuracy substantially over Logistic Regression, which uses the same inputs. **Propose a hypothesis** for *why* (think about what a single set of 6400 weights, one per pixel, can and cannot express, compared with several hidden units that are then combined).
   Then say what **evidence** you could collect to support or refute it. (You do not have to collect it; Optional A below shows one way.)

YOUR ANSWER HERE

---
# Optional extensions

These are optional, but they are how you earn the additional 100 points. They also build intuition.

## Optional A: Look at the first-layer weights

In the Logistic Regression assignment you plotted the learned coefficients as an 80 x 80 image (`model_interpretation`): there is one coefficient per pixel,
so the coefficients form a "template" the model matches.

**Task:**
1. Train a network with one hidden layer of 16 units (`run_experiment([16])`)
2. Extract the weight matrix of its **first** layer: 6400 weights for each of the 16 units
3. Use `helper.showWeights(W)` to show each unit's 6400 weights as an 80 x 80 image (`W` has shape `(6400, 16)`; your toolkit may store the transpose)
4. Also show the Logistic Regression coefficients with `helper.showWeights(lr_clf.coef_[0])`

**Written:** What do you see? Compare with the single Logistic Regression template.
Then: **why does this kind of picture only make sense for the first layer** of a network? What would the weights of the *second* layer be indexed by, and could you reshape them into a picture of pixels?

In [ ]:
# YOUR CODE HERE (optional)

YOUR ANSWER HERE

## Optional B: Non-constant widths ("funnels") and minimum widths

So far every hidden layer had the same width. Try networks whose layers have **different** widths, such as `[2, 256]`, `[256, 2]`, `[4, 256]`, `[256, 4]`, compared with `[2, 2]`, `[4, 4]` and `[256, 256]`.
(Use `run_experiment`, which already accepts any list of widths. Single runs are noisy, so consider averaging a few seeds.)

**Written:** Is there a *minimum* width for a layer, below which making the *other* layers wider doesn't help?
Does it matter whether the narrow layer comes first or later? Why might that be? (Think about what the first layer must do with 6400 inputs, versus what a later layer receives.)

In [ ]:
# YOUR CODE HERE (optional)

YOUR ANSWER HERE

## Optional C: Measure the noise

Repeat the whole grid with **three different seeds** (e.g., `seed=0,1,2` passed to `run_experiment`). Report the **mean and (sample) standard deviation** of validation accuracy for each configuration.
Redo the comparison from Part 3: which of your earlier conclusions survive once you see the spread?

**Written:** (1) List every place in *your* training code where randomness enters. For each, say whether your seed setting controls it, and how you checked (for example, same seed twice, different seeds). (2) In an experiment with "several seeds", should the train/validation split also change from seed to seed? Say what each choice would measure.

In [ ]:
# YOUR CODE HERE (optional)

YOUR ANSWER HERE

## Optional D: Sparse weights with L1 regularization

Logistic Regression's default penalty (L2) shrinks weights but leaves almost all of them nonzero, which is partly why its weight image looks like noise. An **L1** penalty pushes most weights to zero (for Logistic Regression's solver they become exactly zero; with a gradient-based optimizer such as Adam they become tiny but not exactly zero).

**Task:**
1. Fit `LogisticRegression(solver="liblinear", penalty="l1", C=...)` for a few values of `C` (try 0.1, 0.02, 0.005; smaller `C` means a stronger penalty). For each, report the validation accuracy and the fraction of nonzero weights, and show the weight images with `helper.showWeights`. (Recent versions of scikit-learn print a harmless warning about the `penalty` argument.)
2. Add an L1 penalty to the **first layer only** of a network with one hidden layer of 16 units (Keras: `kernel_regularizer=regularizers.l1(lam)`; PyTorch: add `lam * W.abs().sum()` to the loss). Try `lam` in 1e-5, 1e-4, 1e-3. Compare the first-layer weight images and the validation accuracy with the unregularized network. Report the fraction of first-layer weights with absolute value below 0.001. (`helper.showWeights` gives each picture its own color scale, so a unit whose weights are all tiny looks like speckle; pass `shared_scale=True` to compare pictures on one scale.)

**Written:** Does the sparse picture make the "template" easier to read? What does it cost in accuracy? Is a cleaner picture evidence of a better model?

In [ ]:
# YOUR CODE HERE (optional)

YOUR ANSWER HERE

## Optional E: Your own experiments

Anything you are curious about. Some ideas: keep the three color channels instead of gray-scale; try another hidden activation; add dropout; flip or rotate the training images as extra training examples; change the optimizer or learning rate; look at the images the best network gets wrong (error analysis).

For each experiment state a **hypothesis**, change **one thing**, compare with a baseline **while taking the noise into account**, and write a short paragraph on what you found.

In [ ]:
# YOUR CODE HERE (optional)

# Done

In [ ]:
print("Done")